# Day 037 — Exercise 5: clean_dataframe

**What you'll build:** `clean_dataframe(df) -> pd.DataFrame` — a single-call cleaning pipeline that: (1) strips whitespace from all string columns, (2) fills numeric NaN with each column's median, (3) drops duplicate rows and resets the index.

**Why it matters:** Real pipelines need a reliable entry point that normalises any raw DataFrame before analysis. Composing the three techniques into one function gives downstream code a clean guarantee.

## Provided: All Four Cleaning Helpers

In [ ]:
import pandas as pd

def drop_or_fill_nulls(df: pd.DataFrame, strategy: str = 'mean') -> pd.DataFrame:
    result   = df.copy()
    if strategy == 'drop':
        return result.dropna().reset_index(drop=True)
    num_cols = result.select_dtypes(include='number').columns
    if strategy == 'zero':
        result[num_cols] = result[num_cols].fillna(0)
    elif strategy == 'mean':
        for col in num_cols:
            result[col] = result[col].fillna(result[col].mean())
    elif strategy == 'median':
        for col in num_cols:
            result[col] = result[col].fillna(result[col].median())
    else:
        raise ValueError(
            f"Unknown strategy {strategy!r}. "
            "Use 'drop', 'zero', 'mean', or 'median'."
        )
    return result


import pandas as pd

def coerce_numeric_columns(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    result = df.copy()
    for col in columns:
        result[col] = pd.to_numeric(result[col], errors='coerce')
    return result


import pandas as pd

def clean_string_column(df: pd.DataFrame, col: str) -> pd.DataFrame:
    result = df.copy()
    result[col] = result[col].str.strip().str.lower()
    return result


import pandas as pd

def deduplicate(df: pd.DataFrame, subset: list | None = None) -> pd.DataFrame:
    return df.drop_duplicates(subset=subset).reset_index(drop=True)

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd

# MESSY_DF: whitespace in product, one null price, two duplicate rows
MESSY_DF = pd.DataFrame({
    'product': ['  Widget  ', ' Gadget', 'Widget ', '  Widget  ', 'Doohickey'],
    'price':   [25.0, 150.0, 25.0, 25.0, None],
    'qty':     [10.0, 5.0, 10.0, 10.0, 50.0],
})

## Your Implementation

In [ ]:
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Strip → fill nulls → deduplicate.

    1. For every object (string) column: strip leading/trailing whitespace.
    2. For every numeric column: fill NaN with the column median.
    3. Drop duplicate rows and reset the index.
    Original df is not mutated.
    """
    result = df.copy()
    # TODO: for col in result.select_dtypes(include='object').columns:
    #     result[col] = result[col].str.strip()
    # TODO: for col in result.select_dtypes(include='number').columns:
    #     result[col] = result[col].fillna(result[col].median())
    # TODO: return result.drop_duplicates().reset_index(drop=True)
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined, returns DataFrame
    try:
        assert 'clean_dataframe' in globals()
        result = clean_dataframe(MESSY_DF)
        assert isinstance(result, pd.DataFrame), \
            f'expected DataFrame, got {type(result).__name__}'
        passed += 1; print('\u2705 Check 1: returns a DataFrame')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: no numeric nulls remain
    try:
        result = clean_dataframe(MESSY_DF)
        num_nulls = result.select_dtypes(include='number').isnull().sum().sum()
        assert num_nulls == 0, f'{num_nulls} numeric nulls still present'
        passed += 1; print('\u2705 Check 2: no numeric nulls after cleaning')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: string columns have no leading/trailing whitespace
    try:
        result = clean_dataframe(MESSY_DF)
        str_cols = result.select_dtypes(include='object').columns
        for col in str_cols:
            vals = result[col].dropna()
            has_ws = vals.str.startswith(' ') | vals.str.endswith(' ')
            assert not has_ws.any(), \
                f'column {col!r} still has whitespace: {vals[has_ws].tolist()}'
        passed += 1; print('\u2705 Check 3: string columns stripped of whitespace')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: no duplicate rows and index is reset
    try:
        result = clean_dataframe(MESSY_DF)
        assert not result.duplicated().any(), \
            'duplicate rows still present'
        assert list(result.index) == list(range(len(result))), \
            f'index not reset: {list(result.index)}'
        passed += 1; print(f'\u2705 Check 4: no duplicates, index reset ({len(result)} rows)')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: original DataFrame not mutated
    try:
        _ = clean_dataframe(MESSY_DF)
        assert '  Widget  ' in MESSY_DF['product'].values, \
            'original product column was mutated (spaces removed)'
        assert MESSY_DF['price'].isnull().sum() == 1, \
            'original price column was mutated (null filled)'
        passed += 1; print('\u2705 Check 5: original DataFrame not mutated')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
import pandas as pd

def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    for col in result.select_dtypes(include='object').columns:
        result[col] = result[col].str.strip()
    for col in result.select_dtypes(include='number').columns:
        result[col] = result[col].fillna(result[col].median())
    return result.drop_duplicates().reset_index(drop=True)
```

</details>